### Import libraries

In [98]:
import pandas as pd
import numpy as np

### Setting folder path

In [99]:
folder = r"C:\Users\DEBIDO\Desktop\favorita-grocery-sales-forecasting"

### Loading support tables

In [100]:
items = pd.read_csv(folder + r"\items.csv")
stores = pd.read_csv(folder + r"\stores.csv")
transactions = pd.read_csv(folder + r"\transactions.csv")
oil = pd.read_csv(folder + r"\oil.csv")
holidays = pd.read_csv(folder + r"\holidays_events.csv")

### Choosing my project scope

In [101]:
selected_stores = [25, 44, 45, 47, 49]

selected_families = [
    "GROCERY I",
    "BEVERAGES",
    "PRODUCE",
    "DAIRY",
    "CLEANING",
    "BREAD/BAKERY",
    "MEATS",
    "POULTRY"
]

start_date = "2016-01-01"
end_date = "2017-08-15"

### Get selected item numbers

In [102]:
selected_items = items[items["family"].isin(selected_families)]["item_nbr"]

In [103]:
selected_items

0         96995
1         99197
2        103501
3        103520
4        103665
         ...   
4094    2132163
4095    2132318
4096    2132945
4097    2132957
4098    2134058
Name: item_nbr, Length: 3213, dtype: int64

### Loading the train file in simple chunks

In [104]:
filtered_data = []

for chunk in pd.read_csv(folder + r"\train.csv", chunksize=500000, low_memory=False):
    
    chunk["date"] = pd.to_datetime(chunk["date"])
    
    chunk = chunk[
        (chunk["date"] >= start_date) &
        (chunk["date"] <= end_date) &
        (chunk["store_nbr"].isin(selected_stores)) &
        (chunk["item_nbr"].isin(selected_items))
    ]
    
    filtered_data.append(chunk)

train_filtered = pd.concat(filtered_data)

train_filtered.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
66458908,66458908,2016-01-01,25,105574,12.0,False
66458909,66458909,2016-01-01,25,105575,9.0,False
66458910,66458910,2016-01-01,25,105857,3.0,False
66458911,66458911,2016-01-01,25,108634,3.0,False
66458913,66458913,2016-01-01,25,108786,2.0,False


##### I had to load in chunk because the file is too large

In [105]:
train_filtered.shape

(5922789, 6)

### Merge with items and stores

In [106]:
master = train_filtered.merge(items, on="item_nbr", how="left")
master = master.merge(stores, on="store_nbr", how="left")

master.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,state,type,cluster
0,66458908,2016-01-01,25,105574,12.0,False,GROCERY I,1045,0,Salinas,Santa Elena,D,1
1,66458909,2016-01-01,25,105575,9.0,False,GROCERY I,1045,0,Salinas,Santa Elena,D,1
2,66458910,2016-01-01,25,105857,3.0,False,GROCERY I,1092,0,Salinas,Santa Elena,D,1
3,66458911,2016-01-01,25,108634,3.0,False,GROCERY I,1075,0,Salinas,Santa Elena,D,1
4,66458913,2016-01-01,25,108786,2.0,False,CLEANING,3044,0,Salinas,Santa Elena,D,1


### Converting date columns to date datatype

In [107]:
master["date"] = pd.to_datetime(master["date"])
transactions["date"] = pd.to_datetime(transactions["date"])
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])

In [108]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          83488 non-null  datetime64[ns]
 1   store_nbr     83488 non-null  int64         
 2   transactions  83488 non-null  int64         
dtypes: datetime64[ns](1), int64(2)
memory usage: 1.9 MB


In [109]:
oil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        1218 non-null   datetime64[ns]
 1   dcoilwtico  1175 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 19.2 KB


In [110]:
holidays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         350 non-null    datetime64[ns]
 1   type         350 non-null    object        
 2   locale       350 non-null    object        
 3   locale_name  350 non-null    object        
 4   description  350 non-null    object        
 5   transferred  350 non-null    bool          
dtypes: bool(1), datetime64[ns](1), object(4)
memory usage: 14.1+ KB


#### Add transactions table

In [111]:
master = master.merge(transactions, on=["date", "store_nbr"], how="left")

In [112]:
master[["date", "store_nbr", "transactions"]].head()

,date,store_nbr,transactions
0,2016-01-01,25,NaN
1,2016-01-01,25,NaN
2,2016-01-01,25,NaN
3,2016-01-01,25,NaN
4,2016-01-01,25,NaN


#### Add oil price table

In [113]:
master = master.merge(oil, on="date", how="left")
master.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,state,type,cluster,transactions,dcoilwtico
0,66458908,2016-01-01,25,105574,12.0,False,GROCERY I,1045,0,Salinas,Santa Elena,D,1,NaN,NaN
1,66458909,2016-01-01,25,105575,9.0,False,GROCERY I,1045,0,Salinas,Santa Elena,D,1,NaN,NaN
2,66458910,2016-01-01,25,105857,3.0,False,GROCERY I,1092,0,Salinas,Santa Elena,D,1,NaN,NaN
3,66458911,2016-01-01,25,108634,3.0,False,GROCERY I,1075,0,Salinas,Santa Elena,D,1,NaN,NaN
4,66458913,2016-01-01,25,108786,2.0,False,CLEANING,3044,0,Salinas,Santa Elena,D,1,NaN,NaN


### Adding holiday flag

In [114]:
holiday_dates = holidays[holidays["transferred"] == False]["date"]

master["is_holiday"] = master["date"].isin(holiday_dates)

In [115]:
master["is_holiday"].value_counts()

is_holiday
False    4988447
True      934342
Name: count, dtype: int64

### Cleaning missing values

In [123]:
master = master[master["unit_sales"] >= 0].copy()
master = master.reset_index(drop=True)

master["transactions"] = master["transactions"].fillna(0)

master["onpromotion"] = master["onpromotion"].replace({
    "True": True,
    "False": False,
    True: True,
    False: False
})
master["onpromotion"] = master["onpromotion"].fillna(False)
master["onpromotion"] = master["onpromotion"].astype("bool")

master["dcoilwtico"] = master["dcoilwtico"].ffill().bfill()


In [124]:
master["onpromotion"].value_counts()

onpromotion
False    5381136
True      541211
Name: count, dtype: int64

In [125]:
master["onpromotion"].dtype

dtype('bool')

In [126]:
print(master.shape)
print(master.isnull().sum())
master.head()

(5922347, 16)
id              0
date            0
store_nbr       0
item_nbr        0
unit_sales      0
onpromotion     0
family          0
class           0
perishable      0
city            0
state           0
type            0
cluster         0
transactions    0
dcoilwtico      0
is_holiday      0
dtype: int64


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,state,type,cluster,transactions,dcoilwtico,is_holiday
0,66458908,2016-01-01,25,105574,12.0,False,GROCERY I,1045,0,Salinas,Santa Elena,D,1,0.0,36.81,True
1,66458909,2016-01-01,25,105575,9.0,False,GROCERY I,1045,0,Salinas,Santa Elena,D,1,0.0,36.81,True
2,66458910,2016-01-01,25,105857,3.0,False,GROCERY I,1092,0,Salinas,Santa Elena,D,1,0.0,36.81,True
3,66458911,2016-01-01,25,108634,3.0,False,GROCERY I,1075,0,Salinas,Santa Elena,D,1,0.0,36.81,True
4,66458913,2016-01-01,25,108786,2.0,False,CLEANING,3044,0,Salinas,Santa Elena,D,1,0.0,36.81,True


In [127]:
master.shape

(5922347, 16)

In [128]:
master.isnull().sum()

id              0
date            0
store_nbr       0
item_nbr        0
unit_sales      0
onpromotion     0
family          0
class           0
perishable      0
city            0
state           0
type            0
cluster         0
transactions    0
dcoilwtico      0
is_holiday      0
dtype: int64

### Final Master Dataset Summary

The final master dataset was created by filtering the original sales data to selected stores, selected product families, and the most recent historical period.

The dataset includes sales, product details, store details, transaction volume, oil prices, holiday flags, and date-based features.

Missing promotion values were filled as `False`, assuming that missing promotion records indicate no active promotion.

This dataset will be used for exploratory analysis, demand forecasting, inventory optimization, and Power BI dashboard development.

### Saving the final master dataset

In [129]:
master.to_csv(folder + r"\final_master_dataset.csv", index=False)

In [130]:
print("Final master dataset saved successfully")
print("Rows and columns:", master.shape)
print("Date range:", master["date"].min(), "to", master["date"].max())
print("Stores:", master["store_nbr"].nunique())
print("Product families:", master["family"].nunique())
print("Missing values:", master.isnull().sum().sum())

Final master dataset saved successfully
Rows and columns: (5922347, 16)
Date range: 2016-01-01 00:00:00 to 2017-08-15 00:00:00
Stores: 5
Product families: 8
Missing values: 0
